*imports*

In [ ]:

import os, time, json
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree, connected_components
from scipy.linalg import svd
from scipy.spatial import cKDTree

# ------------------- Configuration (edit here) -------------------

### Parameter Tuning Guide

<div style="direction:rtl;text-align:right;font-family:B Lotus, B Nazanin, Tahoma">

* If too many points are added:
<br />
 decrease `PAIR_MAX_DIST_FACTOR` and `SEG_DIST_FACTOR` or increase `MIN_BRIDGE_CLUSTER_PTS` from 1 to 2 or 3.
<br/>
* If no points are bridged but you expect some: slightly increase `PAIR_MAX_DIST_FACTOR` and/or `SEG_DIST_FACTOR` (e.g. from 3 to 4).
<br/> 
*  For a detailed log of what was added, inspect `stats['recover_info']` and check `recovered_indices_in_pts_den`.
<br/>


In [ ]:
INPUT_CSV = "6.csv"
OUTDIR = "results_parsa"
OUTNAME = "6_Out_1.csv"

USE_ZSCORE = True
ZSCORE_THRESH = 5.0

USE_DBSCAN = False
DBSCAN_EPS = 1
DBSCAN_MIN_SAMPLES = 2

K_DENSITY = 8
KEEP_PERCENT = 87.0

K_GRAPH = 4
EDGE_THRESH_MODE = 'percentile'
EDGE_THRESH_VAL = 99.5
DIST_FACTOR = 4.0
MIN_COMP_SIZE = 16
LARGE_COMP_MIN = 30

N_JOBS = 1
SAVE_CURVE = False
DENSIFY_SAMPLES = 600
MIN_PTS_FOR_SPLINE = 4

# Bridge recovery parameters (tunable)
PAIR_MAX_DIST_FACTOR = 3.0    # proximity threshold to cluster cores (factor * median_nn)
SEG_DIST_FACTOR = 2.0         # distance threshold to segment (factor * median_nn)
MIN_BRIDGE_CLUSTER_PTS = 1    # min bridge cluster members for acceptance
MAX_ADDED_PER_PAIR = 200      # max added points per component pair


## --------------------------- Helper functions ----------------------------------------


In [ ]:

def load_csv_xyz(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input file not found: {path}")
    df = pd.read_csv(path, comment='#')
    cols = df.columns.tolist()
    if set(['X','Y','Z']).issubset(set(cols)):
        arr = df[['X','Y','Z']].values
    else:
        arr = df.iloc[:, :3].values
    return arr.astype(float), df

def save_csv_xyz(path, pts, header="X,Y,Z"):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    np.savetxt(path, pts, delimiter=",", header=header, comments='')

def save_curve_obj(path, points, name='curve'):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, 'w') as f:
        f.write(f"# OBJ polyline {name}\n")
        for p in points:
            f.write(f"v {float(p[0])} {float(p[1])} {float(p[2])}\n")
        idxs = " ".join(str(i) for i in range(1, len(points)+1))
        f.write(f"l {idxs}\n")

## --------------------------- Z-score filter ------------------------------------


In [ ]:
def zscore_filter(points, thresh=4.0):
    mu = points.mean(axis=0)
    sigma = points.std(axis=0)
    sigma[sigma==0] = 1e-8
    z = np.abs((points - mu) / sigma)
    mask = np.all(z <= thresh, axis=1)
    return points[mask], mask

## --------------------------- DBSCAN ------------------------------------------


In [ ]:
def dbscan_filter(points, eps=0.5, min_samples=6):
    if len(points) == 0:
        return points, np.zeros(0, dtype=bool)
    db = DBSCAN(eps=eps, min_samples=min_samples)
    labels = db.fit_predict(points)
    mask = labels != -1
    return points[mask], mask


## --------------------------- kNN graph & MST edges -----------------------------


In [ ]:
def build_knn_graph(points, k=8, n_jobs=1):
    n = len(points)
    if n == 0:
        return csr_matrix((n,n))
    k_use = min(k+1, n)
    nn = NearestNeighbors(n_neighbors=k_use, algorithm='kd_tree', n_jobs=n_jobs).fit(points)
    dists, idxs = nn.kneighbors(points)
    rows, cols, data = [], [], []
    for i in range(n):
        for j in range(1, idxs.shape[1]):   # skip self
            rows.append(i); cols.append(int(idxs[i,j])); data.append(float(dists[i,j]))
    A = csr_matrix((data, (rows, cols)), shape=(n,n))
    A = (A + A.T) / 2.0
    return A

def mst_edges_from_adj(A):
    if A.nnz == 0:
        return []
    mst = minimum_spanning_tree(A)
    mst = mst.tocoo()
    edges = list(zip(mst.row.tolist(), mst.col.tolist(), mst.data.tolist()))
    normalized = []
    for u,v,w in edges:
        u=int(u); v=int(v); w=float(w)
        if u< v:
            normalized.append((u,v,w))
        else:
            normalized.append((v,u,w))
    normalized = list({(u,v,w) for (u,v,w) in normalized})
    return normalized


## --------------------------- Cut long edges and remove small components -------------------


In [ ]:
def cut_long_edges_and_components(points, k_graph=12,
                                  edge_thresh_mode='percentile', edge_thresh_val=95.0,
                                  dist_factor=6.0, min_comp_size=10, large_comp_min=30, n_jobs=1):
    info = {}
    n = len(points)
    if n == 0:
        return np.zeros(0, dtype=bool), info
    A = build_knn_graph(points, k=k_graph, n_jobs=n_jobs)
    edges = mst_edges_from_adj(A)
    if len(edges) == 0:
        return np.ones(n, dtype=bool), info
    weights = np.array([w for (_,_,w) in edges], dtype=float)
    info['mst_edge_count'] = int(len(weights))
    if edge_thresh_mode == 'percentile':
        thresh = float(np.percentile(weights, edge_thresh_val))
    elif edge_thresh_mode == 'median_std':
        thresh = float(np.median(weights) + edge_thresh_val * np.std(weights))
    else:
        thresh = float(edge_thresh_val)
    info['edge_thresh'] = thresh
    rows, cols, data = [], [], []
    for (u,v,w) in edges:
        if w <= thresh:
            rows.append(u); cols.append(v); data.append(w)
            rows.append(v); cols.append(u); data.append(w)
    if len(rows) == 0:
        labels = np.arange(n)
        sizes = np.bincount(labels)
        info['pruned_edge_count'] = 0
    else:
        B = csr_matrix((data, (rows, cols)), shape=(n,n))
        n_comp, labels = connected_components(B, directed=False, connection='weak')
        sizes = np.bincount(labels)
        info['pruned_edge_count'] = int(len(data)//2)
        info['n_components'] = int(n_comp)
    centroids = np.zeros((len(sizes),3))
    for i in range(len(sizes)):
        idx = np.where(labels == i)[0]
        if len(idx)>0:
            centroids[i] = points[idx].mean(axis=0)
        else:
            centroids[i] = np.array([np.nan,np.nan,np.nan])
    large_idxs = np.where(sizes >= large_comp_min)[0]
    if len(large_idxs)==0:
        order = np.argsort(sizes)[::-1]
        large_idxs = order[:2] if len(order)>1 else order[:1]
    info['large_idxs'] = large_idxs.tolist()
    nn = NearestNeighbors(n_neighbors=min(2, max(2,n)), algorithm='kd_tree').fit(points)
    dists, _ = nn.kneighbors(points)
    median_nn = float(np.median(dists[:, -1]))
    info['median_nn'] = median_nn
    dist_thresh = dist_factor * median_nn
    info['dist_thresh'] = dist_thresh
    keep_comp = np.ones(len(sizes), dtype=bool)
    for comp in range(len(sizes)):
        if sizes[comp] >= min_comp_size:
            keep_comp[comp] = True
            continue
        if len(large_idxs)>0:
            dists_to_large = np.linalg.norm(centroids[large_idxs] - centroids[comp], axis=1)
            dmin = float(np.min(dists_to_large))
        else:
            dmin = float('inf')
        if dmin > dist_thresh:
            keep_comp[comp] = False
        else:
            keep_comp[comp] = True
    mask_keep = np.array([keep_comp[l] for l in labels], dtype=bool)
    info['component_sizes'] = sizes.tolist()
    info['kept_components'] = int(np.sum(keep_comp))
    info['total_components'] = int(len(sizes))
    return mask_keep, info


## --------------------------- Point-to-segment distance (vectorized) ---------------------


In [ ]:
def point_segment_distance(points, a, b):
    """
    points: (K,3), a,b: (3,) segment endpoints
    returns: distances (K,)
    """
    v = b - a
    vv = np.dot(v, v)
    if vv == 0.0:
        return np.linalg.norm(points - a, axis=1)
    t = np.dot(points - a, v) / vv
    t = np.clip(t, 0.0, 1.0)
    proj = a + np.outer(t, v)
    return np.linalg.norm(points - proj, axis=1)

## --------------------------- Bridge recovery between components -----------------------


In [ ]:

def recover_between_components(pts_den, pts_final,
                               k_graph=K_GRAPH,
                               pair_max_dist_factor=PAIR_MAX_DIST_FACTOR,
                               seg_dist_factor=SEG_DIST_FACTOR,
                               min_bridge_cluster_pts=MIN_BRIDGE_CLUSTER_PTS,
                               max_added_per_pair=MAX_ADDED_PER_PAIR,
                               verbose=True):
    """
    pts_den: points before pruning (after density filter)
    pts_final: points retained after pruning (subset of pts_den)
    Returns candidate points from pts_den/pts_final that lie between components.
    """
    if len(pts_final) == 0:
        return pts_final.copy(), 0, {}
    # 1) Build components on pts_final (kNN graph)
    M = len(pts_final)
    k_use = min(k_graph+1, max(2, M))
    nbr = NearestNeighbors(n_neighbors=k_use, algorithm='kd_tree').fit(pts_final)
    d_temp, idx_temp = nbr.kneighbors(pts_final)
    rows, cols, data = [], [], []
    for i in range(M):
        for j in range(1, idx_temp.shape[1]):
            rows.append(i); cols.append(int(idx_temp[i,j])); data.append(float(d_temp[i,j]))
    A = csr_matrix((data,(rows,cols)), shape=(M,M))
    A = (A + A.T)/2.0
    n_comp, labels_final = connected_components(A, directed=False, connection='weak')
    sizes = np.bincount(labels_final)
    centroids = np.zeros((len(sizes),3))
    for i in range(len(sizes)):
        idx = np.where(labels_final==i)[0]
        if len(idx)>0:
            centroids[i] = pts_final[idx].mean(axis=0)
    # median_nn based on pts_final
    if len(pts_final) > 1:
        nn2 = NearestNeighbors(n_neighbors=2, algorithm='kd_tree').fit(pts_final)
        d2, _ = nn2.kneighbors(pts_final)
        median_nn = float(np.median(d2[:,1]))
    else:
        median_nn = np.mean(np.linalg.norm(pts_final - pts_final.mean(axis=0), axis=1))
    base_scale = median_nn if median_nn>0 else 1.0
    end_thresh = pair_max_dist_factor * base_scale
    seg_thresh = seg_dist_factor * base_scale

    # removed candidates = pts_den that are not in pts_final
    # We must find mapping of pts_final within pts_den; do kd-tree equality or nearest with 0 distance
    tree_den = cKDTree(pts_den)
    d_f_to_den, idx_f_to_den = tree_den.query(pts_final, k=1)
    # mark which indices in pts_den correspond to pts_final (distance ~0)
    mask_den_kept = np.zeros(len(pts_den), dtype=bool)
    for j, dd in enumerate(d_f_to_den):
        if dd <= 1e-9:
            mask_den_kept[int(idx_f_to_den[j])] = True
    removed_idx = np.where(~mask_den_kept)[0]
    if len(removed_idx) == 0:
        if verbose:
            print("[recover_between] no removed candidates")
        return pts_final.copy(), 0, {'added':0}
    removed_pts = pts_den[removed_idx]

    if verbose:
        print(f"[recover_between] comps={len(sizes)}, removed_candidates={len(removed_pts)}, median_nn={median_nn:.6g}")

    # pick large components (cores) to consider connecting
    large_idxs = np.where(sizes >= max(1, int(MIN_COMP_SIZE/2)))[0]  # conservative fallback
    if len(large_idxs) == 0:
        order = np.argsort(sizes)[::-1]
        large_idxs = order[:2] if len(order)>1 else order[:1]

    recovered_indices = set()

    # For each pair of components among the *kept* components (limit pair count for speed)
    comp_ids = list(np.unique(labels_final))
    # optionally restrict to large components + their neighbors; but we'll loop over all pairs but limited
    max_pairs = 2000
    pair_count = 0
    for i in range(len(comp_ids)):
        for j in range(i+1, len(comp_ids)):
            pair_count += 1
            if pair_count > max_pairs:
                break
            ci = comp_ids[i]; cj = comp_ids[j]
            idxs_ci = np.where(labels_final==ci)[0]
            idxs_cj = np.where(labels_final==cj)[0]
            if len(idxs_ci)==0 or len(idxs_cj)==0:
                continue
            # find closest pair between the two components (fast)
            pts_ci = pts_final[idxs_ci]
            pts_cj = pts_final[idxs_cj]
            tree_ci = cKDTree(pts_ci)
            dtmp, itmp = tree_ci.query(pts_cj, k=1)
            jmin = np.argmin(dtmp)
            p1 = pts_ci[int(itmp[jmin])]
            p2 = pts_cj[int(jmin)]
            # now select removed_pts that are within end_thresh to both components' centroids or p1/p2
            # compute dist to p1 and p2
            d1 = np.linalg.norm(removed_pts - p1, axis=1)
            d2 = np.linalg.norm(removed_pts - p2, axis=1)
            both_close = (d1 <= end_thresh) & (d2 <= end_thresh)
            if not np.any(both_close):
                # fallback: allow points close to the segment (but not both ends) but with smaller seg_thresh
                seg_dists = point_segment_distance(removed_pts, p1, p2)
                cand_mask = seg_dists <= seg_thresh
                if not np.any(cand_mask):
                    continue
                cand_idxs = np.where(cand_mask)[0]
            else:
                cand_idxs = np.where(both_close)[0]
            # further filter by distance-to-segment
            cand_pts = removed_pts[cand_idxs]
            seg_dists = point_segment_distance(cand_pts, p1, p2)
            final_mask = seg_dists <= seg_thresh
            sel_local = cand_idxs[final_mask]
            if sel_local.size == 0:
                continue
            # cluster sel_local by DBSCAN (eps ~ 1.5*base_scale) to avoid singletons if desired
            if min_bridge_cluster_pts > 1:
                cluster_eps = max(1e-9, 1.5 * base_scale)
                if len(sel_local) >= 1:
                    db = DBSCAN(eps=cluster_eps, min_samples=1).fit(removed_pts[sel_local])
                    labs = db.labels_
                    for lab in np.unique(labs):
                        mems = np.where(labs==lab)[0]
                        if len(mems) >= min_bridge_cluster_pts:
                            # accept up to max_added_per_pair
                            count_added = 0
                            for m in mems:
                                if len(recovered_indices) >= max_added_per_pair:
                                    break
                                recovered_indices.add(int(removed_idx[sel_local[m]]))
                                count_added += 1
            else:
                # accept all sel_local but cap per pair
                to_add = sel_local.tolist()
                capped = to_add[:max_added_per_pair]
                for loc in capped:
                    recovered_indices.add(int(removed_idx[loc]))

    # build final pts_extended: pts_final + recovered removed_pts (in original order relative to pts_den)
    if len(recovered_indices) == 0:
        if verbose:
            print("[recover_between] no points recovered")
        return pts_final.copy(), 0, {'added':0}
    recovered_list = sorted(list(recovered_indices))
    recovered_pts = pts_den[recovered_list]
    # Merge: keep original pts_final order, then append recovered (or we can merge sorted by projection)
    pts_extended = np.vstack([pts_final, recovered_pts])
    if verbose:
        print(f"[recover_between] recovered {len(recovered_pts)} points (from removed {len(removed_pts)})")
    stats = {'added': int(len(recovered_pts)), 'recovered_indices_in_pts_den': recovered_list}
    return pts_extended, len(recovered_pts), stats


## --------------------------- Simple curve fitting (PCA + poly) --------------------


In [ ]:

def fit_curve_pca_poly(pts, degree=1, n_samples=600):
    if pts.shape[0] < MIN_PTS_FOR_SPLINE:
        return None
    mean = pts.mean(axis=0)
    X = pts - mean
    U,S,Vt = svd(X, full_matrices=False)
    axis1 = Vt.T[:,0]
    axis2 = Vt.T[:,1] if Vt.shape[0] > 1 else np.zeros(3)
    axis3 = np.cross(axis1, axis2)
    B = np.column_stack([axis1, axis2, axis3])
    try:
        Q,R = np.linalg.qr(B)
        B = Q
    except Exception:
        B = np.eye(3)
    coords = X.dot(B)
    t = coords[:,0]; y = coords[:,1]; z = coords[:,2]
    deg = max(1,int(degree))
    coeff_y = np.polyfit(t, y, deg)
    coeff_z = np.polyfit(t, z, deg)
    t_s = np.linspace(t.min(), t.max(), n_samples)
    y_s = np.polyval(coeff_y, t_s)
    z_s = np.polyval(coeff_z, t_s)
    curve_coords = np.vstack([t_s, y_s, z_s]).T
    curve_world = curve_coords.dot(B.T) + mean
    return curve_world


## --------------------------- Main pipeline execution flow ---------------------------


In [ ]:
def run_prune_pipeline_with_recover(input_csv=INPUT_CSV, outdir=OUTDIR, outname=OUTNAME,
                                   use_zscore=USE_ZSCORE, z_thresh=ZSCORE_THRESH,
                                   use_dbscan=USE_DBSCAN, db_eps=DBSCAN_EPS, db_min=DBSCAN_MIN_SAMPLES,
                                   k_density=K_DENSITY, keep_percent=KEEP_PERCENT,
                                   k_graph=K_GRAPH, edge_thresh_mode=EDGE_THRESH_MODE, edge_thresh_val=EDGE_THRESH_VAL,
                                   dist_factor=DIST_FACTOR, min_comp_size=MIN_COMP_SIZE, large_comp_min=LARGE_COMP_MIN,
                                   n_jobs=N_JOBS, save_curve=SAVE_CURVE):
    t0 = time.time()
    pts, df = load_csv_xyz(input_csv)
    n0 = len(pts)
    print(f"[RUN] loaded {n0} points from {input_csv}")
    # 1) Z-score
    mask_z = np.ones(n0, dtype=bool)
    if use_zscore:
        pts_z, mask_z_local = zscore_filter(pts, thresh=z_thresh)
        mask_z = mask_z_local
        print(f"[Z-SCORE] kept {len(pts_z)} / {n0} (thresh={z_thresh})")
    else:
        pts_z = pts.copy()
        print("[Z-SCORE] skipped")
    # 2) DBSCAN (optional)
    mask_db = np.ones(len(pts_z), dtype=bool)
    if use_dbscan:
        pts_db, mask_db_local = dbscan_filter(pts_z, eps=db_eps, min_samples=db_min)
        mask_db = mask_db_local
        print(f"[DBSCAN] kept {len(pts_db)} / {len(pts_z)} (eps={db_eps}, min_samples={db_min})")
    else:
        pts_db = pts_z
        print("[DBSCAN] skipped")
    # 3) density keep
    n_intermediate = len(pts_db)
    if n_intermediate == 0:
        raise RuntimeError("No points remain; relax the filter parameters.")
    k_use = min(k_density+1, max(2, n_intermediate))
    nn = NearestNeighbors(n_neighbors=k_use, algorithm='kd_tree', n_jobs=n_jobs).fit(pts_db)
    dists, _ = nn.kneighbors(pts_db)
    kth = dists[:, -1]
    thresh = float(np.percentile(kth, keep_percent))
    mask_density = kth <= thresh
    pts_den = pts_db[mask_density]
    print(f"[DENSITY] kept {len(pts_den)} / {len(pts_db)} (keep_percent={keep_percent}, kth-thresh={thresh:.6g})")
    if len(pts_den) == 0:
        raise RuntimeError("All points removed by density filter; use gentler settings.")
    # 4) MST-prune + small-component removal
    mask_keep, info_prune = cut_long_edges_and_components(pts_den, k_graph=k_graph,
                                                          edge_thresh_mode=edge_thresh_mode, edge_thresh_val=edge_thresh_val,
                                                          dist_factor=dist_factor, min_comp_size=min_comp_size,
                                                          large_comp_min=large_comp_min, n_jobs=n_jobs)
    pts_final = pts_den[mask_keep]
    print(f"[PRUNE] after MST-prune + small-component removal: kept {len(pts_final)} / {len(pts_den)}")
    print("PRUNE INFO:", json.dumps(info_prune, indent=2))
    # 5) recover-between-components (add only inter-cluster candidate points)
    pts_extended, added_count, recover_stats = recover_between_components(
        pts_den, pts_final,
        k_graph=k_graph,
        pair_max_dist_factor=PAIR_MAX_DIST_FACTOR,
        seg_dist_factor=SEG_DIST_FACTOR,
        min_bridge_cluster_pts=MIN_BRIDGE_CLUSTER_PTS,
        max_added_per_pair=MAX_ADDED_PER_PAIR,
        verbose=True
    )
    print(f"[RECOVER] added {added_count} points between components")
    # final stats
    total_removed = n0 - len(pts_final)
    print("=== SUMMARY ===")
    print("initial_count:", n0)
    print("after_zscore_count:", int(np.sum(mask_z)))
    print("after_dbscan_count:", int(np.sum(mask_db)) if use_dbscan else "skipped")
    print("after_density_count:", len(pts_den))
    print("after_prune_count:", len(pts_final))
    print("after_recover_count:", len(pts_extended))
    print("total_removed (vs initial):", n0 - len(pts_extended))
    # save final
    os.makedirs(outdir, exist_ok=True)
    outpath = os.path.join(outdir, outname)
    save_csv_xyz(outpath, pts_extended)
    t1 = time.time()
    stats = {
        'n_input': n0,
        'n_after_zscore': int(np.sum(mask_z)),
        'n_after_dbscan': (int(np.sum(mask_db)) if use_dbscan else None),
        'n_after_density': len(pts_den),
        'n_after_prune': len(pts_final),
        'n_after_recover': len(pts_extended),
        'added_by_recover': added_count,
        'prune_info': info_prune,
        'recover_info': recover_stats,
        'run_time_s': (t1 - t0)
    }
    print("[RUN] saved final pruned+recovered points ->", outpath)
    # optionally fit curve (ask user)
    try:
        deg_choice = int(input("Fit a curve? Degree (0=no, 1,2,3) default=1: ").strip() or "1")
    except:
        deg_choice = 1
    if deg_choice not in (0,1,2,3):
        deg_choice = 1
    if deg_choice > 0 and len(pts_extended) >= MIN_PTS_FOR_SPLINE:
        print(f"[FIT] fitting degree {deg_choice} polynomial via PCA+poly ...")
        curve = fit_curve_pca_poly(pts_extended, degree=deg_choice, n_samples=DENSIFY_SAMPLES)
        if curve is not None:
            curve_csv = os.path.join(outdir, os.path.splitext(outname)[0] + "_curve.csv")
            curve_obj = os.path.join(outdir, os.path.splitext(outname)[0] + "_curve.obj")
            save_csv_xyz(curve_csv, curve)
            save_curve_obj(curve_obj, curve)
            print("[FIT] saved curve CSV ->", curve_csv)
            print("[FIT] saved curve OBJ ->", curve_obj)
        else:
            print("[FIT] Not enough points for fitting.")
    else:
        print("[FIT] skipped (degree=0 or too few points)")
    return outpath, pts_extended, stats


## --------------------------- Direct sample execution --------------------------------


In [ ]:
if __name__ == "__main__":
    outpath, pts_final, stats = run_prune_pipeline_with_recover(
        input_csv=INPUT_CSV, outdir=OUTDIR, outname=OUTNAME,
        use_zscore=USE_ZSCORE, z_thresh=ZSCORE_THRESH,
        use_dbscan=USE_DBSCAN, db_eps=DBSCAN_EPS, db_min=DBSCAN_MIN_SAMPLES,
        k_density=K_DENSITY, keep_percent=KEEP_PERCENT,
        k_graph=K_GRAPH, edge_thresh_mode=EDGE_THRESH_MODE, edge_thresh_val=EDGE_THRESH_VAL,
        dist_factor=DIST_FACTOR, min_comp_size=MIN_COMP_SIZE, large_comp_min=LARGE_COMP_MIN,
        n_jobs=N_JOBS, save_curve=SAVE_CURVE
    )
    print("DONE. STATS:", json.dumps(stats, indent=2))
